Aprendizado de variedades (Manifold Learning) com fluxo de clusterização distribuída por tipo de característica é mais informativo do que o UMAP para conjuntos de dados clínicos tabulares

Importando as Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
import umap.umap_ as umap
%config InlineBackend.figure_format = 'svg'

In [ ]:
from sklearn import metrics
from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score

In [ ]:
from cluster_val import *

Importando os dados

In [ ]:
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
data=pd.read_csv('cirrhosis.csv',index_col='ID')

In [ ]:
np.random.seed(42)
data=data.sample(frac=1) #Emabaralha o dataset
np.random.seed(42)
i=[x for x in range(418)]

data.set_index(pd.Series(i), inplace=True)

Pré-processamento dos dados

In [ ]:
data.isna().sum()

In [ ]:
data['Status'].value_counts()

In [ ]:
Status_mod= {'Status': {'C':0,'D':1,'CL':2}}
data.replace(Status_mod,inplace=True)
data['Status']

In [ ]:
data['Drug'].value_counts()

In [ ]:
Drug_mod= {'Drug':{'D-penicillamine':0,'Placebo':1}}
data.replace(Drug_mod,inplace=True)
data['Drug']

In [ ]:
data['Sex'].value_counts()

In [ ]:
Sex_mod= {'Sex':{'F':0,'M':1}}
data.replace(Sex_mod,inplace=True)
data['Sex']

In [ ]:
data['Ascites'].value_counts()

In [ ]:
Ascites_mod= {'Ascites':{'N':0,'Y':1}}
data.replace(Ascites_mod,inplace=True)
data['Ascites']

In [ ]:
data['Hepatomegaly'].value_counts()

In [ ]:
Hepatomegaly_mod= {'Hepatomegaly':{'N':0,'Y':1}}
data.replace(Hepatomegaly_mod,inplace=True)
data['Hepatomegaly']

In [ ]:
data['Spiders'].value_counts()

In [ ]:
Spiders_mod= {'Spiders':{'N':0,'Y':1}}
data.replace(Spiders_mod,inplace=True)
data['Spiders']

In [ ]:
data['Edema'].value_counts()

In [ ]:
Edema_mod= {'Edema':{'N':0,'S':1,'Y':2}}
data.replace(Edema_mod,inplace=True)
data['Edema']

Importando valores que faltam

In [ ]:
dense_data_pool=list(data.isna().sum().index[data.isna().sum()<1])
dense_data=data[dense_data_pool]
dense_data=dense_data.dropna()
data=data.loc[np.array(dense_data.index)]

In [ ]:
from scipy.spatial import distance
distance_matrix=[]
for i in range(len(np.array(dense_data))):
    dist=[]
    for j in range(len(np.array(dense_data))):
        d=distance.euclidean(np.array(dense_data)[i],np.array(dense_data)[j])
        dist.append(d)
        neb_list=np.array(dense_data.index)[np.argsort(dist)]
    distance_matrix.append(neb_list)
distance_matrix=np.array(distance_matrix)

In [ ]:
missing_value_list = [x for x in list(data.columns) if x not in dense_data_pool]

In [ ]:
total_impute_master=[]
for f in range(len(missing_value_list)):
    missing_value_indices=data[data[missing_value_list[f]].isnull()].index.tolist()
    feature_impute_master=[]
    for index in missing_value_indices:
        index_in_dist_mat=np.where(distance_matrix[:,0]==index)[0][0]
        value_list=[]
        neb_index_counter=1
        while len(value_list)<6:
            neb_index=distance_matrix[index_in_dist_mat][neb_index_counter]
            neb_index_counter=neb_index_counter+1
            impute_value=data.loc[[neb_index]][missing_value_list[f]]
            if float(impute_value)!=float(impute_value):
                pass
            else:
                value_list.append(float(impute_value))
        feature_impute_master.append(np.array(value_list))
    total_impute_master.append(np.array(feature_impute_master))
total_impute_mater=np.array(total_impute_master)

In [ ]:
total_impute=[]
for i in range(len(total_impute_master)):
    feature_impute=[]
    for j in range(len(total_impute_master[i])):
        intcounter=0
        for k in range(len(total_impute_master[i][j])):
            if total_impute_master[i][j][k]-int(total_impute_master[i][j][k])==0:
                intcounter=intcounter+1
        if intcounter==len(total_impute_master[i][j]):
            imputed_value=np.bincount(total_impute_master[i][j].astype(int)).argmax()
        else:
            imputed_value=np.mean(total_impute_master[i][j])
        feature_impute.append(np.array(imputed_value))
    total_impute.append(np.array(feature_impute))

In [ ]:
for f in range(len(missing_value_list)):
    missing_value_indices=data[data[missing_value_list[f]].isnull()].index.tolist()
    for i in range(len(missing_value_indices)):
        data.at[missing_value_indices[i], missing_value_list[f]]=total_impute[f][i]

In [ ]:
data.isna().sum()

In [ ]:
data=data.drop(['Stage'], axis=1)

In [ ]:
data.shape

UMAP nos dados originais

In [ ]:
from fdc.fdc import feature_clustering
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
from fdc.clustering import *
modified_can = canberra_modified

In [ ]:
umap_emb=feature_clustering(15,0.1,'euclidean',data,True)

Silhouette_score e Dunn index para os clusters do UMAP extraidos usando clustering K-means

In [ ]:
umap_clustering=Clustering(umap_emb,umap_emb,True)
umap_cluster_list,umap_cluster_counts=umap_clustering.K_means(3)

In [ ]:
silhouette_score(umap_emb, umap_cluster_list, metric='euclidean')

Visualizar Silhouette score (Pode-se escolher o números de clusters baseados no resultado)

In [ ]:
Silhouette_visual(umap_emb)

Elbow plot para o embedding

In [ ]:
elbow_plot(umap_emb)

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list))

Silhouette_score e Dunn index para os clusters UMAP extraídos usando clustering aglomerativo

In [ ]:
umap_cluster_list_agglo,umap_cluster_counts_agglo=umap_clustering.Agglomerative(3,'euclidean','ward')

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_agglo))

Silhouette_score e Dunn index para os clusters UMAP extraídos usando clustering do DBSCAN

In [ ]:
umap_cluster_list_dbscan,umap_cluster_counts_dbscan=umap_clustering.DBSCAN(0.5,20)

In [ ]:
#Remove ruído
non_noise_indices= np.where(np.array(umap_cluster_list_dbscan)!=-1)
umap_emb= umap_emb.iloc[non_noise_indices]
#FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
umap_cluster_list_dbscan= np.array(umap_cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(umap_emb, umap_cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(umap_emb,umap_cluster_list_dbscan))

Dividindo as variaveis
- cont_list = continuas
- ord_list = ordinais
-  nom_list  = nominais

In [ ]:
cont_list= ['N_Days','Age','Bilirubin','Cholesterol','Albumin','Copper','Alk_Phos','SGOT','Tryglicerides','Platelets','Prothrombin']

ord_list= ['Ascites','Hepatomegaly','Spiders','Edema']

nom_list= ['Status','Drug','Sex']

In [ ]:
len(ord_list)

In [ ]:
len(nom_list)

In [ ]:
len(cont_list)

FDC nos dados originais

In [ ]:
from fdc.fdc import feature_clustering
from fdc.fdc import FDC, Clustering
from fdc.fdc import canberra_modified
modified_can = canberra_modified

In [ ]:
fdc = FDC(clustering_cont=Clustering('euclidean',15,0.1)
          , clustering_ord=Clustering(modified_can,15,0.1)
          , clustering_nom=Clustering('hamming',15,0.1)
          , visual=True
          , use_pandas_output=True
          , with_2d_embedding=True
          )

fdc.selectFeatures(continueous=cont_list, nomial=nom_list, ordinal=ord_list)

FDC_emb_high,FDC_emb_low = fdc.normalize(data,n_neighbors=15, min_dist=0.1,cont_list=cont_list, nom_list=nom_list, ord_list=ord_list,
                  with_2d_embedding=True,
                  visual=True)

Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando k-means

In [ ]:
from fdc.clustering import Clustering

clustering=Clustering(FDC_emb_high,FDC_emb_low,True)
cluster_list,cluster_counts=clustering.K_means(3)

In [ ]:
FDC_emb_high['Cluster'] = cluster_list

In [ ]:
silhouette_score(FDC_emb_high, cluster_list, metric='euclidean')

In [ ]:
Silhouette_visual(FDC_emb_high)

In [ ]:
elbow_plot(FDC_emb_high)

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list))

Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando clustering aglomerativo

In [ ]:
cluster_list_agglo,cluster_counts_agglo=clustering.Agglomerative(3,'euclidean','ward')

In [ ]:
FDC_emb_high['Cluster'] = cluster_list_agglo

In [ ]:
silhouette_score(FDC_emb_high, cluster_list_agglo, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list_agglo))

Silhouette_score e Dunn index para os clusters FDC (de dimensão intermediaria) extraídos usando clustering do DBSCAN

In [ ]:
cluster_list_dbscan,cluster_counts_dbscan=clustering.DBSCAN(1.5,15)

In [ ]:
FDC_emb_high['Cluster'] = cluster_list_dbscan

In [ ]:
#Remove ruído
non_noise_indices= np.where(np.array(cluster_list_dbscan)!=-1)
FDC_emb_high= FDC_emb_high.iloc[non_noise_indices]
FDC_emb_low= FDC_emb_low.iloc[non_noise_indices]
cluster_list_dbscan= np.array(cluster_list_dbscan)[non_noise_indices]

In [ ]:
silhouette_score(FDC_emb_high, cluster_list_dbscan, metric='euclidean')

In [ ]:
dunn_index(cluster_wise_df(FDC_emb_high,cluster_list_dbscan))